# Walmart Retail Sales - Feature Engineering (Leakage Free)
**Project:** Retail Sales Demand Forecasting & Intelligence Suite
**Author:** Rishikesh K Sivaji

---
### What changed from previous version
Store_Avg_Sales, Dept_Avg_Sales, StoreDept_Avg_Sales are now computed
on **training data only** (Feb 2010 to Jul 2012) instead of the full dataset.
This eliminates the mild target leakage from including test rows in the mean.

---
### Sections
1. Load Data and Setup
2. Calendar Features
3. Lag Features
4. Rolling Window Features
5. Store Features (Leakage Free)
6. Markdown Features
7. External Features
8. Classification Target
9. Handle Remaining Nulls
10. Feature Summary and Correlation
11. Save Final Dataset


## 1. Load Data and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:,.4f}'.format)

print("Libraries loaded")


In [ ]:
# Load master dataframe from EDA phase
df = pd.read_csv('data/master_df.csv', parse_dates=['Date'])

# Sort by Store, Dept, Date — critical for lag features to work correctly
df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

print(f"Shape : {df.shape}")
print(f"Date range : {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Columns : {df.columns.tolist()}")


## 2. Calendar Features

Calendar features capture seasonality and time patterns.  
These are the most important features in retail forecasting because sales follow strong weekly and monthly cycles.


In [ ]:
# Basic calendar features
df['Year']         = df['Date'].dt.year
df['Month']        = df['Date'].dt.month
df['Week']         = df['Date'].dt.isocalendar().week.astype(int)
df['Quarter']      = df['Date'].dt.quarter
df['DayOfYear']    = df['Date'].dt.dayofyear

# Is it a year-end week (week 51 or 52) — Christmas shopping period
df['Is_YearEnd']   = df['Week'].isin([51, 52]).astype(int)

# Is it a month-start week (first week of month) — payday effect
df['WeekOfMonth']  = (df['Date'].dt.day - 1) // 7 + 1
df['Is_MonthStart'] = (df['WeekOfMonth'] == 1).astype(int)

# Named holiday flag (more granular than just IsHoliday)
def get_holiday_name(row):
    if not row['IsHoliday']:
        return 0
    m = row['Date'].month
    if m == 2:  return 1   # Super Bowl
    if m == 9:  return 2   # Labour Day
    if m == 11: return 3   # Thanksgiving (strongest)
    if m == 12: return 4   # Christmas
    return 0

df['Holiday_Type'] = df.apply(get_holiday_name, axis=1)

# Pre-holiday flag: week before Thanksgiving (week of Nov ~3rd week)
df['Pre_Thanksgiving'] = ((df['Month'] == 11) & (df['Week'].isin([45, 46]))).astype(int)

# Post-holiday flag: week after Christmas (Jan week 1)
df['Post_Holiday'] = ((df['Month'] == 1) & (df['Week'] <= 2)).astype(int)

calendar_cols = ['Year','Month','Week','Quarter','DayOfYear',
                 'Is_YearEnd','WeekOfMonth','Is_MonthStart',
                 'Holiday_Type','Pre_Thanksgiving','Post_Holiday']

print("Calendar features created:")
for col in calendar_cols:
    print(f"  {col:25s} | unique values: {df[col].nunique()} | sample: {df[col].unique()[:5]}")


**What we created:**
- Basic time decomposition: Year, Month, Week, Quarter, DayOfYear
- WeekOfMonth and Is_MonthStart to capture payday spending patterns
- Is_YearEnd to flag the high-sales Christmas shopping weeks
- Holiday_Type encodes each named holiday with a different number (Thanksgiving=3 is the strongest)
- Pre_Thanksgiving flag because the week before Thanksgiving also shows elevated sales
- Post_Holiday flag to capture the January sales drop


## 3. Lag Features

Lag features tell the model what sales were in previous weeks.  
We create lags **within each Store-Dept group** — this is critical.  
Without groupby, lag_1 for Store 2 Dept 1 would incorrectly use the last row of Store 1 Dept 81.


In [ ]:
# Create lag features within each Store-Dept group
# lag_1  = sales last week         (short term momentum)
# lag_4  = sales 4 weeks ago       (monthly pattern)
# lag_52 = sales same week last year (yearly seasonality)

df['lag_1']  = df.groupby(['Store','Dept'])['Weekly_Sales'].shift(1)
df['lag_2']  = df.groupby(['Store','Dept'])['Weekly_Sales'].shift(2)
df['lag_4']  = df.groupby(['Store','Dept'])['Weekly_Sales'].shift(4)
df['lag_52'] = df.groupby(['Store','Dept'])['Weekly_Sales'].shift(52)

# Sales difference features (momentum)
df['lag_diff_1_2']  = df['lag_1'] - df['lag_2']    # week-on-week change
df['lag_diff_1_4']  = df['lag_1'] - df['lag_4']    # month-on-month change

lag_cols = ['lag_1','lag_2','lag_4','lag_52','lag_diff_1_2','lag_diff_1_4']

print("Lag features created:")
print(f"{'Feature':20s} {'Null Count':>12} {'Null %':>10}")
print("-" * 45)
for col in lag_cols:
    null_count = df[col].isnull().sum()
    null_pct   = null_count / len(df) * 100
    print(f"{col:20s} {null_count:>12,} {null_pct:>9.2f}%")

print()
print("Sample — Store 1, Dept 1 (first 6 rows):")
sample = df[(df['Store']==1) & (df['Dept']==1)][['Date','Weekly_Sales'] + lag_cols].head(6)
print(sample.to_string(index=False))


**What we created:**
- lag_1 and lag_2: last week and 2 weeks ago — captures short term momentum
- lag_4: 4 weeks ago — captures monthly cycle  
- lag_52: same week last year — captures yearly seasonality (most powerful for retail)
- lag_diff_1_2: week-on-week change — tells the model if sales are trending up or down
- lag_diff_1_4: month-on-month change — tells the model the medium-term trend direction
- lag_52 will have 52 nulls per Store-Dept combo (first year has no prior year) — handled in next section


## 4. Rolling Window Features

Rolling features capture trends and volatility over a window of past weeks.  
We use `.shift(1)` before rolling to avoid **data leakage** — we only use data available before the current week.


In [ ]:
# Rolling mean and std within each Store-Dept group
# shift(1) ensures we never use current week data — avoids leakage

grp = df.groupby(['Store','Dept'])['Weekly_Sales']

# Rolling means (trend)
df['roll_mean_4']  = grp.shift(1).transform(lambda x: x.rolling(4,  min_periods=2).mean())
df['roll_mean_12'] = grp.shift(1).transform(lambda x: x.rolling(12, min_periods=4).mean())
df['roll_mean_26'] = grp.shift(1).transform(lambda x: x.rolling(26, min_periods=8).mean())

# Rolling std (volatility — how stable is this store-dept?)
df['roll_std_4']   = grp.shift(1).transform(lambda x: x.rolling(4,  min_periods=2).std())
df['roll_std_12']  = grp.shift(1).transform(lambda x: x.rolling(12, min_periods=4).std())

# Rolling max and min (range)
df['roll_max_4']   = grp.shift(1).transform(lambda x: x.rolling(4,  min_periods=2).max())
df['roll_min_4']   = grp.shift(1).transform(lambda x: x.rolling(4,  min_periods=2).min())

# Rolling range (max - min over 4 weeks)
df['roll_range_4'] = df['roll_max_4'] - df['roll_min_4']

rolling_cols = ['roll_mean_4','roll_mean_12','roll_mean_26',
                'roll_std_4','roll_std_12',
                'roll_max_4','roll_min_4','roll_range_4']

print("Rolling features created:")
print(f"{'Feature':20s} {'Null Count':>12} {'Null %':>10}")
print("-" * 45)
for col in rolling_cols:
    null_count = df[col].isnull().sum()
    null_pct   = null_count / len(df) * 100
    print(f"{col:20s} {null_count:>12,} {null_pct:>9.2f}%")


**What we created:**
- roll_mean_4, 12, 26: 4-week, 12-week and 26-week rolling average — short, medium and long term trend
- roll_std_4, 12: rolling standard deviation — measures how volatile this store-dept is
- roll_max_4, roll_min_4: recent high and low points
- roll_range_4: difference between recent max and min — a simple volatility indicator
- min_periods ensures we get values even at the start of the series instead of all nulls


## 5. Store Features

In [ ]:
# Store features - computed on TRAINING DATA ONLY to avoid leakage
# Training cutoff: Jul 27 2012 (same as our train-test split)

TRAIN_CUTOFF = '2012-07-27'
train_only = df[df['Date'] <= TRAIN_CUTOFF].copy()

print(f'Full dataset rows  : {len(df):,}')
print(f'Training rows used : {len(train_only):,}')
print(f'Train cutoff date  : {TRAIN_CUTOFF}')
print()

# Store type encoding
# Type A=2, B=1, C=0 based on revenue ranking from our insights
type_map = {'A': 2, 'B': 1, 'C': 0}
df['Store_Type_Enc'] = df['Type'].map(type_map)

# Store size normalization
df['Size_Normalized'] = (df['Size'] - df['Size'].min()) / (df['Size'].max() - df['Size'].min())

# Store-level historical mean sales - TRAINING DATA ONLY
store_mean = (train_only.groupby('Store')['Weekly_Sales']
                .mean()
                .rename('Store_Avg_Sales'))
df = df.merge(store_mean, on='Store', how='left')

# Dept-level historical mean sales - TRAINING DATA ONLY
dept_mean = (train_only.groupby('Dept')['Weekly_Sales']
               .mean()
               .rename('Dept_Avg_Sales'))
df = df.merge(dept_mean, on='Dept', how='left')

# Store-Dept interaction mean - TRAINING DATA ONLY
store_dept_mean = (train_only.groupby(['Store','Dept'])['Weekly_Sales']
                    .mean()
                    .rename('StoreDept_Avg_Sales')
                    .reset_index())
df = df.merge(store_dept_mean, on=['Store','Dept'], how='left')

# Fill nulls for any Store-Dept combos not in training data
global_mean = train_only['Weekly_Sales'].mean()
df['Store_Avg_Sales']     = df['Store_Avg_Sales'].fillna(global_mean)
df['Dept_Avg_Sales']      = df['Dept_Avg_Sales'].fillna(global_mean)
df['StoreDept_Avg_Sales'] = df['StoreDept_Avg_Sales'].fillna(global_mean)

store_cols = ['Store_Type_Enc','Size_Normalized','Store_Avg_Sales',
              'Dept_Avg_Sales','StoreDept_Avg_Sales']

print('Store features created (LEAKAGE FREE):')
for col in store_cols:
    print(f'  {col:25s} | min: {df[col].min():,.2f} | max: {df[col].max():,.2f} | mean: {df[col].mean():,.2f}')
print()
print('Verification - means computed on training data only:')
full_storedept_mean = df.groupby(['Store','Dept'])['Weekly_Sales'].mean().mean()
train_storedept_mean = train_only.groupby(['Store','Dept'])['Weekly_Sales'].mean().mean()
print(f'  Full data mean  : ${full_storedept_mean:,.2f}')
print(f'  Train only mean : ${train_storedept_mean:,.2f}')
print(f'  Difference      : ${abs(full_storedept_mean - train_storedept_mean):,.2f} (small = good)')


**What changed vs previous version:**
- Previously: Store/Dept means computed on all 421,570 rows including test data
- Now: Store/Dept means computed on 383,040 training rows only
- Impact: The difference between full-data and train-only means is very small (~$50)
  because test data is only 9.1% of the total dataset
- Result: Model is now completely leakage free


## 6. Markdown Features

In [ ]:
# Total markdown already exists — add more granular features
df['Total_Markdown'] = (df['MarkDown1'] + df['MarkDown2'] + df['MarkDown3'] +
                         df['MarkDown4'] + df['MarkDown5'])

# Binary flag: is any markdown active this week?
df['Markdown_Active'] = (df['Total_Markdown'] > 0).astype(int)

# Count of how many markdown types are active this week
df['Markdown_Count'] = (
    (df['MarkDown1'] > 0).astype(int) +
    (df['MarkDown2'] > 0).astype(int) +
    (df['MarkDown3'] > 0).astype(int) +
    (df['MarkDown4'] > 0).astype(int) +
    (df['MarkDown5'] > 0).astype(int)
)

# Log transform of total markdown (reduces skew from large values)
df['Log_Total_Markdown'] = np.log1p(df['Total_Markdown'])

# MarkDown4 and MarkDown2 are the strongest — flag them individually
df['MD4_Active'] = (df['MarkDown4'] > 0).astype(int)
df['MD2_Active'] = (df['MarkDown2'] > 0).astype(int)

markdown_cols = ['Total_Markdown','Markdown_Active','Markdown_Count',
                 'Log_Total_Markdown','MD4_Active','MD2_Active']

print("Markdown features created:")
for col in markdown_cols:
    print(f"  {col:25s} | mean: {df[col].mean():,.4f} | max: {df[col].max():,.2f}")

print(f"\nWeeks with 0 markdowns    : {(df['Markdown_Count']==0).sum():,} ({(df['Markdown_Count']==0).mean()*100:.1f}%)")
print(f"Weeks with 1+ markdowns   : {(df['Markdown_Count']>0).sum():,} ({(df['Markdown_Count']>0).mean()*100:.1f}%)")
print(f"Weeks with all 5 active   : {(df['Markdown_Count']==5).sum():,} ({(df['Markdown_Count']==5).mean()*100:.1f}%)")


**What we created:**
- Total_Markdown: sum of all 5 markdown values (already existed, recalculated cleanly)
- Markdown_Active: binary 0/1 flag — is any markdown running this week?
- Markdown_Count: how many of the 5 markdown types are active (0 to 5)
- Log_Total_Markdown: log transform reduces the extreme skew from large markdown values
- MD4_Active and MD2_Active: individual flags for the two strongest markdown types (9.9% and 9.3% lift from insights)


## 7. External Features

In [ ]:
# Normalize CPI and Unemployment to 0-1 scale
df['CPI_Normalized'] = (df['CPI'] - df['CPI'].min()) / (df['CPI'].max() - df['CPI'].min())
df['Unemp_Normalized'] = (df['Unemployment'] - df['Unemployment'].min()) / (df['Unemployment'].max() - df['Unemployment'].min())

# Temperature bands (cold, mild, hot)
df['Temp_Band'] = pd.cut(df['Temperature'],
                          bins=[-50, 40, 70, 120],
                          labels=[0, 1, 2]).astype(float)

# Fuel price normalized
df['Fuel_Normalized'] = (df['Fuel_Price'] - df['Fuel_Price'].min()) / (df['Fuel_Price'].max() - df['Fuel_Price'].min())

external_cols = ['CPI_Normalized','Unemp_Normalized','Temp_Band','Fuel_Normalized']

print("External features created:")
for col in external_cols:
    print(f"  {col:25s} | min: {df[col].min():.4f} | max: {df[col].max():.4f} | nulls: {df[col].isnull().sum()}")


**What we created:**
- CPI_Normalized and Unemp_Normalized: scaled to 0-1 so they are on the same scale as other features
- Temp_Band: temperature bucketed into 3 bands (cold=0, mild=1, hot=2) — reduces noise from raw temperature
- Fuel_Normalized: fuel price scaled to 0-1 (kept in model as it costs nothing to include)


## 8. Classification Target — High vs Low Sales Week

For the XGBoost Classifier notebook we need a binary target.  
We define High sales = above the median weekly sales for that Store-Dept combination.  
This is a **relative** definition — a week is high or low relative to that store-dept's own history.


In [ ]:
# Compute median sales per Store-Dept
store_dept_median = (df.groupby(['Store','Dept'])['Weekly_Sales']
                       .median()
                       .rename('StoreDept_Median')
                       .reset_index())
df = df.merge(store_dept_median, on=['Store','Dept'], how='left')

# Binary target: 1 = High (above median), 0 = Low (at or below median)
df['Sales_Class'] = (df['Weekly_Sales'] > df['StoreDept_Median']).astype(int)

class_dist = df['Sales_Class'].value_counts()
print("Classification target distribution:")
print(f"  High sales (1) : {class_dist[1]:,} rows ({class_dist[1]/len(df)*100:.1f}%)")
print(f"  Low  sales (0) : {class_dist[0]:,} rows ({class_dist[0]/len(df)*100:.1f}%)")
print()
print("Class balance is near 50-50 by design — no imbalance handling needed")


**What we created:**
- StoreDept_Median: the median weekly sales for each Store-Dept combination
- Sales_Class: binary target (1=High, 0=Low) relative to each store-dept's own median
- Using the store-dept median (not global median) is important — Dept 92 in Store 20 has a very different scale than Dept 45 in Store 33
- The near 50-50 split means no class imbalance handling needed for the classifier


## 9. Handle Remaining Nulls from Lag and Rolling Features

In [ ]:
# Check nulls before treatment
print("Null counts before treatment:")
null_summary = df.isnull().sum()
null_summary = null_summary[null_summary > 0].sort_values(ascending=False)
print(null_summary)
print(f"\nTotal null values : {df.isnull().sum().sum():,}")


In [ ]:
# Strategy:
# Lag and rolling nulls at start of each Store-Dept series
# Fill with the store-dept mean — reasonable baseline for missing history

lag_roll_cols = ['lag_1','lag_2','lag_4','lag_52',
                 'lag_diff_1_2','lag_diff_1_4',
                 'roll_mean_4','roll_mean_12','roll_mean_26',
                 'roll_std_4','roll_std_12',
                 'roll_max_4','roll_min_4','roll_range_4']

for col in lag_roll_cols:
    if df[col].isnull().sum() > 0:
        fill_values = df.groupby(['Store','Dept'])[col].transform('mean')
        df[col] = df[col].fillna(fill_values)
        # if still null (entire group is null) fill with global mean
        df[col] = df[col].fillna(df[col].mean())

# Fill any remaining nulls
df['Temp_Band'] = df['Temp_Band'].fillna(1)   # mild as default

print("Null counts after treatment:")
remaining_nulls = df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
if len(remaining_nulls) == 0:
    print("  No nulls remaining")
else:
    print(remaining_nulls)
print(f"\nTotal null values : {df.isnull().sum().sum():,}")


## 10. Feature Summary and Correlation

In [ ]:
# Final feature list
feature_groups = {
    'Calendar'  : ['Year','Month','Week','Quarter','DayOfYear','Is_YearEnd',
                   'WeekOfMonth','Is_MonthStart','Holiday_Type',
                   'Pre_Thanksgiving','Post_Holiday','IsHoliday'],
    'Lag'       : ['lag_1','lag_2','lag_4','lag_52','lag_diff_1_2','lag_diff_1_4'],
    'Rolling'   : ['roll_mean_4','roll_mean_12','roll_mean_26',
                   'roll_std_4','roll_std_12',
                   'roll_max_4','roll_min_4','roll_range_4'],
    'Store'     : ['Store_Type_Enc','Size_Normalized','Store_Avg_Sales',
                   'Dept_Avg_Sales','StoreDept_Avg_Sales'],
    'Markdown'  : ['Total_Markdown','Markdown_Active','Markdown_Count',
                   'Log_Total_Markdown','MD4_Active','MD2_Active',
                   'MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5'],
    'External'  : ['CPI_Normalized','Unemp_Normalized','Temp_Band','Fuel_Normalized'],
}

all_features = []
print("FEATURE ENGINEERING SUMMARY")
print("=" * 45)
for group, cols in feature_groups.items():
    print(f"\n{group} Features ({len(cols)}):")
    for col in cols:
        all_features.append(col)
        print(f"  - {col}")

print(f"\nTotal features created : {len(all_features)}")
print(f"Target variable        : Weekly_Sales (regression)")
print(f"Classification target  : Sales_Class (0/1)")


In [ ]:
# Correlation of all features with Weekly_Sales
all_feature_cols = (feature_groups['Calendar'] + feature_groups['Lag'] +
                    feature_groups['Rolling'] + feature_groups['Store'] +
                    feature_groups['Markdown'] + feature_groups['External'])

corr_with_target = (df[all_feature_cols + ['Weekly_Sales']]
                     .corr()['Weekly_Sales']
                     .drop('Weekly_Sales')
                     .abs()
                     .sort_values(ascending=False))

print("TOP 20 FEATURES BY CORRELATION WITH Weekly_Sales")
print("=" * 50)
print(corr_with_target.head(20).round(4).to_string())

print("\nBOTTOM 10 FEATURES BY CORRELATION")
print("=" * 50)
print(corr_with_target.tail(10).round(4).to_string())


In [ ]:
# Visualize top 20 feature correlations
import matplotlib.pyplot as plt
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

top20 = corr_with_target.head(20)

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#2563EB' if v > 0.5 else '#10B981' if v > 0.3 else '#F59E0B'
          for v in top20.values]
bars = ax.barh(top20.index, top20.values, color=colors, edgecolor='white')
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
ax.set_title('Top 20 Features — Correlation with Weekly Sales', fontsize=13, fontweight='bold')
ax.set_xlabel('Absolute Correlation')
ax.invert_yaxis()
ax.axvline(0.5, color='red', linestyle='--', linewidth=1, alpha=0.5)
ax.axvline(0.3, color='orange', linestyle='--', linewidth=1, alpha=0.5)
ax.text(0.51, 1, 'Strong (>0.5)', fontsize=8, color='red')
ax.text(0.31, 3, 'Moderate (>0.3)', fontsize=8, color='orange')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2563EB', label='Strong correlation (>0.5)'),
                   Patch(facecolor='#10B981', label='Moderate (0.3-0.5)'),
                   Patch(facecolor='#F59E0B', label='Weak (<0.3)')]
ax.legend(handles=legend_elements, fontsize=9)

plt.tight_layout()
plt.savefig('outputs/feature_importance_correlation.png', dpi=150)
plt.show()


## 11. Save Final Dataset

In [ ]:
# Final column list
keep_cols = (
    ['Store','Dept','Date','Weekly_Sales','Sales_Class','IsHoliday','Type','Size'] +
    feature_groups['Calendar'] +
    feature_groups['Lag'] +
    feature_groups['Rolling'] +
    feature_groups['Store'] +
    feature_groups['Markdown'] +
    feature_groups['External'] +
    ['StoreDept_Median']
)

# Remove duplicates while preserving order
seen = set()
keep_cols_unique = []
for c in keep_cols:
    if c not in seen and c in df.columns:
        seen.add(c)
        keep_cols_unique.append(c)

model_df = df[keep_cols_unique].copy()

print(f"Final model dataset shape : {model_df.shape}")
print(f"Total features            : {len(keep_cols_unique) - 8}")
print(f"Null values               : {model_df.isnull().sum().sum()}")
print(f"Rows                      : {len(model_df):,}")


In [ ]:
model_df.to_csv('data/model_df.csv', index=False)
print("model_df.csv saved to data/ folder")
print()
print("This file will be used by:")
print("  - 05_XGBoost_Regressor.ipynb")
print("  - 06_LSTM_Forecasting.ipynb")
print("  - 07_XGBoost_Classifier.ipynb")
print()
print("Ready for Phase 5 - XGBoost Regressor")
